# 🪖 ดาวน์โหลดแพ็กเกจติดตั้งสำหรับ Raspberry Pi 4 (Offline Installer Generator)

สมุดบันทึกนี้ออกแบบมาเพื่อดาวน์โหลดไฟล์ `.whl` สำหรับนำไปติดตั้งบน **Raspberry Pi 4 แบบ Offline (ไม่ง้อเน็ตในบอร์ด)** ผ่าน Flash Drive

> ⚠️ **จุดสังเกตสำคัญเรื่อง OS ของ Pi 4:**
> เครื่อง Raspberry Pi OS ส่วนใหญ่ (รวมถึงเครื่องคุณ) จะเป็น **32-bit (armv7l, Python 3.9)** หรือบางคนใช้ **64-bit (aarch64)**
> สคริปต์ด้านล่างนี้เตรียมให้ครบทั้ง **Pure NCNN (แนะนำที่สุด ขนาด 5MB)** และ **Ultralytics ตัวเต็ม** พร้อมแยกหมวดหมู่อย่างถูกต้อง

## ⚡ ขั้นตอนที่ 1: เตรียมโฟลเดอร์สำหรับเก็บแพ็กเกจ

In [ ]:
!rm -rf pi_packages pi_packages.zip
!mkdir -p pi_packages/ncnn_armv7l_32bit
!mkdir -p pi_packages/ncnn_aarch64_64bit
!mkdir -p pi_packages/ultralytics_full
print("✅ สร้างโฟลเดอร์สำหรับดาวน์โหลดเรียบร้อย!")

## 🚀 ขั้นตอนที่ 2: โหลด Pure NCNN Engine (แนะนำอันนี้! 5MB รันเร็ว 35+ FPS)
โหลดไฟล์ `.whl` ของ NCNN สำหรับทั้ง 32-bit (armv7l cp39) และ 64-bit (aarch64)

In [ ]:
# 1. โหลด NCNN สำหรับ 32-bit (armv7l, Python 3.9 - ตรงกับเครื่องคุณเป๊ะๆ)
!pip download ncnn \
    --platform manylinux_2_31_armv7l \
    --python-version 39 \
    --implementation cp \
    --abi cp39 \
    --only-binary=:all: \
    --no-deps \
    -d ./pi_packages/ncnn_armv7l_32bit

# 2. โหลด NCNN สำหรับ 64-bit (aarch64, Python 3.9 และ 3.11) เผื่อสลับ OS
!pip download ncnn \
    --platform manylinux2014_aarch64 \
    --python-version 39 \
    --implementation cp \
    --abi cp39 \
    --only-binary=:all: \
    --no-deps \
    -d ./pi_packages/ncnn_aarch64_64bit || true

print("\n✅ ดาวน์โหลด NCNN Engine เรียบร้อย!")

## 📦 ขั้นตอนที่ 3: โหลด Ultralytics ตัวเต็ม (เผื่อต้องการใช้งาน)
ดาวน์โหลดแพ็กเกจ Ultralytics พร้อม Dependencies ที่ซัพพอร์ต ARM

In [ ]:
# ดาวน์โหลด wheel สำหรับ armv7l และ aarch64
!pip download ultralytics \
    --extra-index-url https://www.piwheels.org/simple \
    --prefer-binary \
    -d ./pi_packages/ultralytics_full || true

print("\n✅ ตรวจสอบไฟล์ในโฟลเดอร์ pi_packages:")
!ls -lh pi_packages/*

## 💾 ขั้นตอนที่ 4: บีบอัดเป็น ZIP แล้วดาวน์โหลดลงคอมพิวเตอร์
รันช่องนี้แล้ว Colab จะเด้งดาวน์โหลดไฟล์ `pi_packages.zip` ให้อัตโนมัติ

In [ ]:
# สร้างไฟล์ readme สรุปวิธีติดตั้งข้างใน zip
readme_text = """# 🪖 วิธีติดตั้งแพ็กเกจแบบ Offline บน Raspberry Pi 4

1. สำหรับวิธีที่เร็วที่สุด (Pure NCNN):
   - เข้าโฟลเดอร์ ncnn_armv7l_32bit (ถ้าใช้ Pi OS 32-bit):
     cd ncnn_armv7l_32bit
     pip3 install *.whl --no-deps
   - จากนั้นรันระบบได้เลย: ./start_pi4.sh

2. สำหรับ Ultralytics ตัวเต็ม:
   - เข้าโฟลเดอร์ ultralytics_full:
     cd ultralytics_full
     pip3 install *.whl --no-index --find-links .
"""
with open("pi_packages/HOW_TO_INSTALL.txt", "w", encoding="utf-8") as f:
    f.write(readme_text)

# Zip และดาวน์โหลด
!zip -r pi_packages.zip pi_packages

from google.colab import files
files.download('pi_packages.zip')
print("\n🎉 กำลังส่งไฟล์ pi_packages.zip ไปที่คอมพิวเตอร์ของคุณ!")